### Imports

In [ ]:
import warnings 
warnings.filterwarnings("ignore") 
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
import pandas as pd 
nltk.download('punkt')
import re

In [ ]:
# reading the radiology data
df = pd.read_csv(r"C:\Users\Varshni\Documents\Semester 2\ST5230\Assignment 1\radiology.csv\radiology.csv")
df.head()

# Data Preprocessing

## We see that the text column contains details such as Examination, Indication, Technique, Comparison, Findings, Impression

## So, we extract the details and store them in separate columns

In [ ]:
# defining patterns to extract
patterns = {
    'Examination': r"EXAMINATION:\s*(\w+)",   
    'Indication': r"INDICATION:\s*(.+?)(?:\n|$)",
    'Technique': r"TECHNIQUE:\s*(.+?)(?:\n|$)",
    'Comparison': r"COMPARISON:\s*(.+?)(?:\n|$)",
    'Findings': r"FINDINGS:\s*(.+?)(?:\n|$)",
    'Impression': r"IMPRESSION:\s*(.+?)(?:\n|$)"
}

# extracting the required fields
for col, pattern in patterns.items():
    df[col] = df['text'].str.extract(pattern)

# dropping the text column 
df = df.drop("text", axis = 1)

df.head()

# This dataset contains a lot of noise in the form of special characters, improper formatting, etc. making it unsuitable as a model input.
## So, we clean the data through the following steps:
### 1. Convert it to lowercase 
### 2. Remove special characters 
### 3. Tokenize the text

In [ ]:
# defining the list of columns to be preprocessed 
column_list = ['Examination', 'Indication', 'Technique','Comparison', 'Findings', 'Impression']

# combining the relevant columns
df['combined_text'] = df[column_list].astype(str).apply(lambda x: ' '.join(x), axis=1)

# Text Preprocessing Function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\W+', ' ', text)  # Remove special characters
    tokens = word_tokenize(text)  # Tokenize
    return tokens

# Apply preprocessing
df['tokenized_text'] = df['combined_text'].apply(preprocess_text)
df.head()

## To further remove the noise, we remove stop words such as 'is', 'were', etc.

In [ ]:
from gensim.parsing.preprocessing import remove_stopwords

# removing stop words 
df['tokenized_text'] = df['tokenized_text'].apply(lambda x: remove_stopwords(' '.join(x)).split())
df.head()

In [ ]:
# remove words such as 'nan', 'year', 'old' from the tokenized text
words_to_remove = ['nan', 'year', 'old']
df['tokenized_text'] = df['tokenized_text'].apply(
    lambda x: [word for word in x if word not in words_to_remove]
)

df.head()

## Based on words such as "woman", "female" and "man", "male", we assign gender as F or M

In [ ]:
female_indicators = ['woman', 'female']
male_indicators = ['man', 'male']

df['Gender'] = df['tokenized_text'].apply(
    lambda x: 'F' if any(word in x for word in female_indicators) 
              else ('M' if any(word in x for word in male_indicators) else None)
)

df.head()

### Exporting the preprocessed data as a CSV

In [ ]:
# exporting the data for further usage
df.to_csv("Preprocessed_radiology_data.csv", index = False) 